In [1]:
import pygeoinf as inf 
import numpy as np
import pandas as pd
from typing import List, Any
import matplotlib.pyplot as plt
from aux import GridSpace, WaterMassForwardOperator, create_spatial_covariance_matrix, create_point_measurement_operator
from data_helper import load_em_matrix, get_measurement_locations

In [2]:
lat_min = -90
lat_max = -30
lon_min = -180
lon_max = 180
degree_resolution = 4

def lat_to_y(lat):
    return (lat - lat_min) / degree_resolution

def lon_to_x(lon):
    return (lon - lon_min) / degree_resolution

def y_to_lat(y):
    return y * degree_resolution + lat_min

def x_to_lon(x):
    return x * degree_resolution + lon_min

In [3]:
nx = int((lon_max - lon_min) / degree_resolution) + 1
ny = int((lat_max - lat_min) / degree_resolution) + 1

grid = GridSpace((ny, nx))

mat, order = load_em_matrix()

In [4]:
op1 = WaterMassForwardOperator(grid, mat).as_linear_operator()

lats, lons = get_measurement_locations()
number_of_measurements = len(lats)
ys = [lat_to_y(lat) for lat in lats]
xs = [lon_to_x(lon) for lon in lons]
op2 = create_point_measurement_operator(xs, ys, op1.codomain)

forward_operator = op2 @ op1

single_variable_measurement_space = forward_operator.codomain.subspaces[0]
data_error_measure = inf.GaussianMeasure.from_standard_deviation(single_variable_measurement_space, 0.001)
joint_data_error_measure = inf.GaussianMeasure.from_direct_sum([data_error_measure] * forward_operator.codomain.number_of_subspaces)

forward_problem = inf.LinearForwardProblem(forward_operator, data_error_measure=joint_data_error_measure)

In [5]:
prior_covariance_matrix = create_spatial_covariance_matrix(grid.shape, 0.3, 4.0, True)
prior_expectation = np.full(grid.shape,  0.25)
prior_measure = inf.GaussianMeasure.from_covariance_matrix(grid, prior_covariance_matrix, expectation=prior_expectation)
joint_prior_measure = inf.GaussianMeasure.from_direct_sum([prior_measure] * forward_operator.domain.number_of_subspaces)

np.random.seed(76)
true_model, data = forward_problem.synthetic_model_and_data(joint_prior_measure)

# Set the last component of data to an array of ones as a conservation of mass constraint
data[-1] = np.ones(number_of_measurements)

In [6]:
#=======================================================================
# Perform the Bayesian inversion to get the posterior measure
#=======================================================================

# --- Create the preconditioner ---
preconditioner = inf.SpectralPreconditioningMethod(
    rank=250,
    method='variable', 
    power=2,
    max_rank=500,
    parallel=True,
    n_jobs=10
)

# --- Pass it into the matrix-free Conjugate Gradient solver ---
solver = inf.CGSolver(
    preconditioning_method=preconditioner,
    rtol=1e-4,
    callback=inf.ProgressCallback("CG Iteration: ")
)

# --- Set up the Bayesian inversion ---
bayesian_inversion = inf.LinearBayesianInversion(
    forward_problem,
    joint_prior_measure
)
# --- Solve for the posterior measure ---
model_posterior_measure = bayesian_inversion.model_posterior_measure(
    data,
    solver
)

KeyboardInterrupt: 

In [ ]:
eigenvalues = np.linalg.eigvalsh(prior_covariance_matrix)
print(eigenvalues)